# Fixed-Lag Smoother Example with Smart Factors

A vehicle drives a loop with a forward-looking camera among random landmarks; a landmark is measured when it is in view and the detector does not drop it, so tracks start, pause, and end at random. Each landmark is one `SmartProjectionPoseFactor`. A factor inside the smoother is never modified: a copy with the new measurement replaces it through `factorsToRemove`. A track that outlives the window is marginalized with the leaving pose (`MARGINALIZE`, the default) or continued with that pose held fixed (`CONDITION`).

Both modes run on `BatchFixedLagSmoother` and `IncrementalFixedLagSmoother` and are compared with the full batch solution; see [SmartFactorsFixedLag.md](https://github.com/borglab/gtsam/blob/develop/gtsam/slam/doc/SmartFactorsFixedLag.md). The C++ version is `examples/FixedLagSmootherExample_SmartFactor.cpp`, and `VisualFixedLagSmootherExample_SmartFactor.ipynb` does the same on recorded data.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/nonlinear/FixedLagSmootherExample_SmartFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [2]:
import math

import numpy as np
import plotly.graph_objects as go

import gtsam
from gtsam.symbol_shorthand import X

## 1. Simulation

Sixty frames half a second apart on a loop of radius 25 m, 120 landmarks, one pixel of noise, 30 percent detection dropout, and noisy odometry. `frame_factors` gives the prior or odometry factor of a frame with the initial value of its pose, and `full_batch` is the reference solution over all frames up to a given one.

In [3]:
class Simulation:
    num_frames, frame_interval, num_landmarks, dropout = 60, 0.5, 120, 0.3

    def __init__(self, seed=7):
        rng = np.random.default_rng(seed)
        self.K = gtsam.Cal3_S2(450.0, 450.0, 0.0, 320.0, 240.0)
        self.body_P_camera = gtsam.Pose3(gtsam.Rot3.Ypr(-math.pi / 2, 0.0, -math.pi / 2),
                                         gtsam.Point3(0.2, 0.0, 0.5))  # looks along the body x axis
        self.pixel_noise = gtsam.noiseModel.Isotropic.Sigma(2, 1.0)
        sigmas = np.array([0.004, 0.004, 0.004, 0.02, 0.02, 0.02])
        self.odometry_noise = gtsam.noiseModel.Diagonal.Sigmas(sigmas)
        self.prior_noise = gtsam.noiseModel.Isotropic.Sigma(6, 1e-3)

        # A loop of radius 25 m on the ground, heading along the tangent
        self.truth = []
        for i in range(self.num_frames):
            angle = 2 * math.pi * i / self.num_frames
            self.truth.append(gtsam.Pose3(gtsam.Rot3.Yaw(angle + math.pi / 2),
                                          gtsam.Point3(25 * math.cos(angle), 25 * math.sin(angle), 0.0)))
        # Landmarks 20 to 40 m from the center, 1 to 4 m above the ground
        self.landmarks = []
        for _ in range(self.num_landmarks):
            angle, radius = 2 * math.pi * rng.uniform(), 20 + 20 * rng.uniform()
            self.landmarks.append(gtsam.Point3(radius * math.cos(angle), radius * math.sin(angle),
                                               1 + 3 * rng.uniform()))
        # Seen when in front of the camera within 40 m, detected, and in the image
        self.measurements = []  # per frame: {landmark: pixel}
        for pose in self.truth:
            camera = gtsam.PinholePoseCal3_S2(pose.compose(self.body_P_camera), self.K)
            frame = {}
            for j, landmark in enumerate(self.landmarks):
                local = camera.pose().transformTo(landmark)
                if local[2] < 2.0 or np.linalg.norm(local) > 40.0 or rng.uniform() < self.dropout:
                    continue
                pixel = camera.project(landmark) + rng.normal(size=2)
                if 0 <= pixel[0] <= 640 and 0 <= pixel[1] <= 480:
                    frame[j] = pixel
            self.measurements.append(frame)
        # Odometry with noise
        self.odometry = [None]
        for i in range(1, self.num_frames):
            step = self.truth[i - 1].between(self.truth[i])
            self.odometry.append(step.compose(gtsam.Pose3.Expmap(sigmas * rng.normal(size=6))))

    def new_factor(self):
        return gtsam.SmartProjectionPoseFactorCal3_S2(self.pixel_noise, self.K, self.body_P_camera)

    def frame_factors(self, i, previous, graph, values):
        if i == 0:
            graph.push_back(gtsam.PriorFactorPose3(X(0), self.truth[0], self.prior_noise))
            values.insert(X(0), self.truth[0])
        else:
            graph.push_back(gtsam.BetweenFactorPose3(X(i - 1), X(i), self.odometry[i], self.odometry_noise))
            values.insert(X(i), previous.compose(self.odometry[i]))

    def smart_factors(self, up_to):
        """One smart factor per landmark seen at least twice in frames 0 ... up_to."""
        factors = {}
        for i in range(up_to + 1):
            for j, pixel in self.measurements[i].items():
                factors.setdefault(j, self.new_factor()).add(pixel, X(i))
        return {j: factor for j, factor in factors.items() if factor.size() >= 2}

    def full_batch(self, up_to):
        graph, initial = gtsam.NonlinearFactorGraph(), gtsam.Values()
        for i in range(up_to + 1):
            self.frame_factors(i, initial.atPose3(X(i - 1)) if i > 0 else None, graph, initial)
        for factor in self.smart_factors(up_to).values():
            graph.push_back(factor)
        return gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()

## 2. The replace protocol

A `Track` holds the factor of one landmark and its slot in the smoother, or -1 until it has two measurements. Every frame, a copy of the factor with the new measurement replaces the one in the smoother; the slots of the copies come back in `getNewFactorsIndices()`, followed by the slots of the conditioned copies the smoother made of the factors in `getConditionedFactorIndices()`. A slot that is empty afterwards means the factor was consumed by marginalization; that track, or one seen only once whose pose has since left the window, starts afresh. Every fifth frame the newest pose is compared with the full batch solution. After every update, `run` also records the latest estimate of every pose and the point every smart factor in the smoother triangulates, for the plot in section 4.

In [4]:
class Track:
    def __init__(self):
        self.factor, self.slot = None, -1


def run(smoother, sim):
    tracks, restarts, longest, errors, final = {}, 0, 0, [], 0.0
    trajectory, landmarks = gtsam.Values(), {}
    for i in range(sim.num_frames):
        new_factors, new_values = gtsam.NonlinearFactorGraph(), gtsam.Values()
        sim.frame_factors(i, smoother.calculateEstimatePose3(X(i - 1)) if i > 0 else None,
                          new_factors, new_values)

        to_remove, pending = [], []  # pending: (landmark, index in new_factors)
        graph, timestamps = smoother.getFactors(), smoother.timestamps()
        for j, pixel in sim.measurements[i].items():
            track = tracks.setdefault(j, Track())
            in_smoother = (track.slot >= 0 and graph.exists(track.slot)
                           and graph.at(track.slot).equals(track.factor, 1e-12))
            if track.factor is not None and track.slot < 0 and track.factor.keys()[0] not in timestamps:
                track.factor, restarts = None, restarts + 1
            factor = sim.new_factor() if track.factor is None else track.factor.clone()
            factor.add(pixel, X(i))
            if in_smoother:
                to_remove.append(track.slot)
            track.factor, track.slot = factor, -1
            if factor.size() >= 2:
                pending.append((j, new_factors.size()))
                new_factors.push_back(factor)
        ours = new_factors.size()
        result = smoother.update(new_factors, new_values, {X(i): i * sim.frame_interval}, to_remove)

        new_slots = result.getNewFactorsIndices()
        for j, index in pending:
            tracks[j].slot = new_slots[index]
        by_slot = {track.slot: track for track in tracks.values() if track.slot >= 0}
        for k, old_slot in enumerate(result.getConditionedFactorIndices()):
            if old_slot in by_slot:
                by_slot[old_slot].slot = new_slots[ours + k]
        graph = smoother.getFactors()
        for track in tracks.values():
            if track.slot < 0:
                continue
            factor = graph.at(track.slot) if graph.exists(track.slot) else None
            if isinstance(factor, gtsam.SmartProjectionPoseFactorCal3_S2):
                track.factor = factor
                longest = max(longest, factor.dim() // 2)  # two rows per measurement
            else:
                track.factor, track.slot, restarts = None, -1, restarts + 1

        # The latest estimate of every pose, and the point every smart factor in the smoother
        # triangulates; a copy triangulates, so the smoother's factor keeps its cache.
        estimate = smoother.calculateEstimate()
        trajectory.insert_or_assign(estimate)
        for j, track in tracks.items():
            if track.slot >= 0:
                point = track.factor.clone().point(estimate)
                if point.valid():
                    landmarks[j] = point.get()
        if (i + 1) % 5 == 0:
            estimate = smoother.calculateEstimatePose3(X(i)).translation()
            errors.append(np.linalg.norm(estimate - sim.full_batch(i).atPose3(X(i)).translation()))
            final = np.linalg.norm(estimate - sim.truth[i].translation())
    return dict(mean=np.mean(errors), max=np.max(errors), final=final, restarts=restarts,
                longest=longest, errors=errors, trajectory=trajectory, landmarks=landmarks)

## 3. Both smoothers, both modes

The lag is eight frames. Which mode stays closer to the batch solution depends on the data; `MARGINALIZE` is exact marginalization but ends each track at the window, `CONDITION` keeps whole tracks but freezes each pose's estimation error when it is fixed. The two smoothers agree with each other to a few centimetres in both modes.

In [5]:
sim = Simulation(seed=7)
print(f"{sim.num_frames} frames, {sim.num_landmarks} landmarks, "
      f"{sum(len(frame) for frame in sim.measurements)} measurements")
batch = sim.full_batch(sim.num_frames - 1).atPose3(X(sim.num_frames - 1)).translation()
print(f"Full batch final error to truth: {np.linalg.norm(batch - sim.truth[-1].translation()):.3f} m\n")

lag = 8 * sim.frame_interval  # eight frames
isam_params = gtsam.ISAM2Params()
isam_params.findUnusedFactorSlots = True
isam_params.relinearizeSkip = 1
isam_params.setRelinearizeThreshold(0.01)

print(f"{'smoother / mode':30}{'mean|fl-batch|':16}{'max|fl-batch|':15}{'final|fl-gt|':14}{'restarts':10}longest track")
summaries = {}
modes = gtsam.FixedLagSmoother.MarginalizationMode
for mode_name, mode in [("MARGINALIZE", modes.MARGINALIZE), ("CONDITION", modes.CONDITION)]:
    smoothers = [("Incremental", gtsam.IncrementalFixedLagSmoother(lag, isam_params)),
                 ("Batch", gtsam.BatchFixedLagSmoother(lag))]
    for smoother_name, smoother in smoothers:
        smoother.setMarginalizationMode(mode)
        name = f"{smoother_name} / {mode_name}"
        s = summaries[name] = run(smoother, sim)
        print(f"{name:30}{s['mean']:<16.3f}{s['max']:<15.3f}{s['final']:<14.3f}{s['restarts']:<10}{s['longest']}")

60 frames, 120 landmarks, 808 measurements
Full batch final error to truth: 0.057 m

smoother / mode               mean|fl-batch|  max|fl-batch|  final|fl-gt|  restarts  longest track
Incremental / MARGINALIZE     0.103           0.430          0.478         124       9
Batch / MARGINALIZE           0.104           0.428          0.476         124       9


Incremental / CONDITION       0.115           0.513          0.566         99        11
Batch / CONDITION             0.110           0.480          0.535         99        11


In [6]:
fig = go.Figure()
for name, s in summaries.items():
    fig.add_trace(go.Scatter(x=[5 * (k + 1) - 1 for k in range(len(s["errors"]))], y=s["errors"],
                             name=name, mode="lines+markers"))
fig.update_layout(title="Distance of the newest pose from the full batch solution",
                  xaxis_title="frame", yaxis_title="m", height=400)
fig.show()

## 4. Trajectory and landmarks

The ground truth, the full batch solution, and the trajectory of each smoother and mode, in which every pose keeps its last estimate before it left the window. Smart factors have no landmark variables: a full batch landmark is the point its smart factor triangulates at the solution, and a fixed-lag landmark is the last point its smart factor in the smoother triangulated. Every landmark with an estimate is plotted, and the printed counts name the others: a landmark seen fewer than twice has no estimate. A fixed-lag estimate of a landmark comes from fewer measurements, or from poses fixed at older estimates, than the full batch one, and differs from it by about 0.3 m, so their points overlap only partly. Hover over a point to see its landmark, and click a legend entry to show or hide a trace.

In [7]:
counts = np.bincount([j for frame in sim.measurements for j in frame], minlength=sim.num_landmarks)
print(f"{sim.num_landmarks} landmarks, {np.sum(counts >= 2)} of them seen at least twice; "
      f"never seen: {np.flatnonzero(counts == 0).tolist()}, seen once: {np.flatnonzero(counts == 1).tolist()}")
batch = sim.full_batch(sim.num_frames - 1)
triangulations = {j: factor.point(batch) for j, factor in sim.smart_factors(sim.num_frames - 1).items()}
estimates = {"Full batch": ("black", batch, {j: point.get() for j, point in triangulations.items() if point.valid()})}
for (name, s), color in zip(summaries.items(), ["royalblue", "deepskyblue", "crimson", "orange"]):
    estimates[name] = (color, s["trajectory"], s["landmarks"])

truth = np.array([pose.translation() for pose in sim.truth])
true_landmarks = np.array(sim.landmarks)
fig = go.Figure([
    go.Scatter3d(x=truth[:, 0], y=truth[:, 1], z=truth[:, 2], mode="lines", name="Ground truth",
                 line=dict(color="gray", width=2, dash="dash")),
    go.Scatter3d(x=true_landmarks[:, 0], y=true_landmarks[:, 1], z=true_landmarks[:, 2], mode="markers",
                 name="Ground truth: landmarks", text=[f"landmark {j}" for j in range(sim.num_landmarks)],
                 marker=dict(size=2, color="gray"))])
for name, (color, values, points) in estimates.items():
    ids = sorted(points)
    xyz = np.array([points[j] for j in ids])
    errors = np.linalg.norm(xyz - true_landmarks[ids], axis=1)
    print(f"{name}: {len(ids)} of {sim.num_landmarks} landmarks plotted, no estimate for "
          f"{sorted(set(range(sim.num_landmarks)) - set(ids))}; error to the truth: median {np.median(errors):.2f} m, "
          f"max {errors.max():.2f} m")
    poses = np.array([values.atPose3(X(i)).translation() for i in range(sim.num_frames)])
    fig.add_trace(go.Scatter3d(x=poses[:, 0], y=poses[:, 1], z=poses[:, 2], mode="lines+markers",
                               name=name, line=dict(color=color, width=3), marker=dict(size=2, color=color)))
    fig.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                               name=f"{name}: landmarks", text=[f"landmark {j}" for j in ids],
                               marker=dict(size=2, color=color),
                               visible=True if name == "Full batch" else "legendonly"))
fig.update_layout(title="Trajectory and landmarks", height=600, legend=dict(x=0, y=1),
                  margin=dict(l=0, r=0, b=0, t=30),
                  scene=dict(xaxis_title="x (m)", yaxis_title="y (m)", zaxis_title="z (m)", aspectmode="data"))
fig.show()

120 landmarks, 115 of them seen at least twice; never seen: [12, 62, 65, 117], seen once: [78]
Full batch: 115 of 120 landmarks plotted, no estimate for [12, 62, 65, 78, 117]; error to the truth: median 0.24 m, max 1.58 m
Incremental / MARGINALIZE: 115 of 120 landmarks plotted, no estimate for [12, 62, 65, 78, 117]; error to the truth: median 0.43 m, max 1.65 m
Batch / MARGINALIZE: 115 of 120 landmarks plotted, no estimate for [12, 62, 65, 78, 117]; error to the truth: median 0.43 m, max 1.68 m
Incremental / CONDITION: 115 of 120 landmarks plotted, no estimate for [12, 62, 65, 78, 117]; error to the truth: median 0.49 m, max 1.99 m
Batch / CONDITION: 115 of 120 landmarks plotted, no estimate for [12, 62, 65, 78, 117]; error to the truth: median 0.48 m, max 1.87 m
